[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-10-capstone-ml-pipeline.ipynb#scrollTo=ca101010)

---
# Day 10 · Capstone — End-to-End Distributed ML Pipeline
## Data → Train → Tune → Serve
**certified-journeys / ray-certified** · Exam Day

> **Goal for today:** Build a complete, working distributed ML pipeline using every major Ray library — Ray Data, Ray Train, Ray Tune, and Ray Serve — with clear handoff points documented at each stage.

### Pipeline Overview

```
1. ray.init()          → verify cluster topology
2. Ray Data            → load breast cancer dataset, preprocess with .map_batches()
3. Ray Train           → TorchTrainer with ScalingConfig(num_workers=1)
4. Ray Tune            → search over lr, batch_size, hidden_size (5 trials)
5. Best checkpoint     → load, run inference on test set
6. Ray Serve           → deploy POST /predict endpoint
7. Integration test    → send 10 samples, assert predictions
8. ray.timeline()      → profile and document bottleneck
```


In [ ]:
%pip install -q 'ray[default,train,tune,serve,data]' torch scikit-learn optuna requests


## Stage 1 · Initialize Ray and Verify Cluster Topology

Always start by auditing the cluster's available resources before submitting heavy workloads. For a Colab single-node cluster, verify that CPU count, memory, and object store capacity are sufficient.


In [ ]:
import ray
import json
import time
import os

# Start Ray with enough object store for our tensors (~500 MB)
ray.init(
    ignore_reinit_error=True,
    include_dashboard=True,
    object_store_memory=500 * 1024 * 1024,  # 500 MB
)

# ── Capture topology and log to JSON ─────────────────────────────────────────
cluster_resources = ray.cluster_resources()
available        = ray.available_resources()

topology = {
    "ray_version"      : ray.__version__,
    "total_resources"  : cluster_resources,
    "available"        : available,
    "num_nodes"        : len(ray.nodes()),
}

topology_path = "/tmp/ray_topology.json"
with open(topology_path, "w") as f:
    json.dump(topology, f, indent=2, default=str)

print("Cluster topology:")
print(json.dumps(topology, indent=2, default=str))
print(f"\nTopology saved to: {topology_path}")


### What just happened?
- **`ray.cluster_resources()`** returns the *total* resources registered across all nodes; **`ray.available_resources()`** returns what is currently free.
- Logging topology to JSON gives you an audit trail — useful for debugging "job failed to start" errors where the root cause is insufficient CPUs.
- **`ray.nodes()`** returns one dict per node in the cluster; `len(ray.nodes())` is the node count.
- In a Colab environment you'll typically see `CPU: 2.0` and ~2 GB of available memory.


## Stage 2 · Ray Data — Load and Preprocess the Dataset

**Handoff: Ray Data → Ray Train**
The `Dataset` object returned by Ray Data is passed directly to `TorchTrainer` via the `datasets` argument. Ray Train shards it across workers automatically.

Key Ray Data operations:
- `ray.data.from_items()` — create a dataset from a Python list of dicts
- `.map_batches(fn)` — apply a function to each batch (vectorised preprocessing)
- `.train_test_split(test_size)` — split into train/test without shuffling to memory


In [ ]:
import ray.data
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

# ── Load breast cancer dataset ────────────────────────────────────────────────
bc = load_breast_cancer()
X, y = bc.data.astype(np.float32), bc.target.astype(np.int64)

# ── Fit scaler on full dataset (in practice, fit only on train split) ─────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# ── Convert to list of dicts for Ray Data ─────────────────────────────────────
records = [
    {"features": X_scaled[i].tolist(), "label": int(y[i])}
    for i in range(len(X_scaled))
]

# Create a Ray Dataset — data is stored in the distributed object store
ds = ray.data.from_items(records)
print(f"Dataset: {ds.count()} rows, {len(bc.feature_names)} features")
print(f"Schema: {ds.schema()}")


In [ ]:
import pyarrow as pa

def validate_batch(batch: dict) -> dict:
    """Map function: validate feature range and clip outliers."""
    import numpy as _np
    # batch is a dict of {column_name: np.ndarray} when using map_batches
    features = _np.array(batch["features"])
    # Clip to [-5, 5] standard deviations — removes extreme outliers
    clipped = _np.clip(features, -5.0, 5.0)
    return {"features": clipped.tolist(), "label": batch["label"]}

# Apply preprocessing using map_batches — runs in parallel across the dataset
ds_clean = ds.map_batches(validate_batch, batch_format="numpy")

# Split into train (80%) and test (20%)
# Ray Data's split preserves the distributed nature — no materialisation to driver
train_ds, test_ds = ds_clean.train_test_split(test_size=0.2, shuffle=True, seed=42)
print(f"Train size: {train_ds.count()}, Test size: {test_ds.count()}")

# Peek at a few rows to verify preprocessing
sample = train_ds.take(2)
print("\nSample row (first 5 features):", sample[0]["features"][:5], "label:", sample[0]["label"])


### What just happened?
- **`ray.data.from_items()`** distributes the records across Ray's object store in blocks — no single Python process holds all the data.
- **`.map_batches()`** runs the preprocessing function in parallel across all data blocks; `batch_format="numpy"` gives the function numpy arrays for vectorised operations.
- **`.train_test_split()`** returns two `Dataset` objects — neither is materialised to the driver yet; they're lazy references to partitions.
- **Handoff to Ray Train**: pass `datasets={"train": train_ds, "valid": test_ds}` to `TorchTrainer` — Train handles sharding across workers.


## Stage 3 · Ray Train — Define the TorchTrainer Training Function

**Handoff: Ray Train → Ray Tune**
The `TorchTrainer` becomes the *trainable* passed to `Tuner`. Tune calls it once per trial, injecting hyperparameters via the `config` dict.

Ray Train patterns:
- `train_loop_per_worker(config)` — runs on each worker process
- `train.report({"metric": value})` — sends metrics back to Tune after each epoch
- `train.get_dataset_shard("train")` — gets this worker's shard of the dataset
- `train.get_checkpoint()` — loads a checkpoint (for resume/fine-tuning)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import ray.train as train
from ray.train import ScalingConfig, Checkpoint
from ray.train.torch import TorchTrainer

# ── Model definition ──────────────────────────────────────────────────────────
class MLP(nn.Module):
    """Simple two-layer MLP for binary classification."""

    def __init__(self, input_dim: int, hidden_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 2),  # 2 classes: benign / malignant
        )

    def forward(self, x):
        return self.net(x)


# ── Training loop (runs on every worker) ─────────────────────────────────────
def train_loop_per_worker(config: dict):
    """Called once per worker per trial. config carries hyperparameters."""
    lr          = config["lr"]
    batch_size  = config["batch_size"]
    hidden_size = config["hidden_size"]
    num_epochs  = config.get("num_epochs", 10)
    input_dim   = 30  # breast cancer has 30 features

    # ── Get this worker's dataset shard ──────────────────────────────────────
    train_shard = train.get_dataset_shard("train")

    # Convert Ray Dataset shard to PyTorch tensors
    rows = list(train_shard.iter_rows())
    X_t  = torch.tensor([r["features"] for r in rows], dtype=torch.float32)
    y_t  = torch.tensor([r["label"]    for r in rows], dtype=torch.long)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    # ── Model, optimizer, loss ────────────────────────────────────────────────
    model     = train.torch.prepare_model(MLP(input_dim, hidden_size))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)

        avg_loss = total_loss / total
        accuracy = correct / total

        # Save checkpoint on last epoch
        checkpoint = None
        if epoch == num_epochs - 1:
            checkpoint = Checkpoint.from_dict({
                "model_state": model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
                "hidden_size": hidden_size,
                "input_dim"  : input_dim,
            })

        # Report metrics to Tune / Train
        train.report(
            {"val_loss": avg_loss, "val_accuracy": accuracy},
            checkpoint=checkpoint,
        )


print("Training function defined. MLP architecture verified.")
mlp_test = MLP(30, 64)
x_test = torch.randn(4, 30)
print("Forward pass test:", mlp_test(x_test).shape)  # should be [4, 2]


### What just happened?
- **`train.get_dataset_shard("train")`** gives each worker its own slice of the `train_ds` we created in Stage 2 — no data is duplicated across workers.
- **`train.torch.prepare_model()`** wraps the model for distributed training (DDP) when `num_workers > 1`; with `num_workers=1` it's a no-op.
- **`train.report({...}, checkpoint=...)`** sends metrics to the Tune trial and optionally saves a checkpoint — this is the Ray Train → Ray Tune handoff mechanism.
- **`Checkpoint.from_dict()`** serialises the model weights + metadata to the object store; Tune uses this to find the best trial's checkpoint.


## Stage 4 · Ray Tune — Hyperparameter Search

**Handoff: Ray Tune → best checkpoint**
After `tuner.fit()`, call `tuner.get_results().get_best_result()` to get the best trial's metrics and checkpoint.

Search space:
- `lr`: log-uniform over [1e-4, 1e-2]
- `batch_size`: grid over [16, 32]
- `hidden_size`: choice from [32, 64, 128]

We use `num_samples=5` (Colab-friendly) with `ASHAScheduler` to prune bad trials early.


In [ ]:
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch

# ── Search space ──────────────────────────────────────────────────────────────
search_space = {
    "lr"          : tune.loguniform(1e-4, 1e-2),
    "batch_size"  : tune.grid_search([16, 32]),
    "hidden_size" : tune.choice([32, 64, 128]),
    "num_epochs"  : 8,                            # fixed — not tuned
}

# ASHA prunes unpromising trials after each epoch
scheduler = ASHAScheduler(
    metric="val_accuracy",
    mode="max",
    max_t=8,            # max epochs per trial
    grace_period=3,     # run at least 3 epochs before pruning
    reduction_factor=2,
)

# Optuna's TPE sampler for intelligent search over the continuous lr dimension
searcher = OptunaSearch(metric="val_accuracy", mode="max")

# ── Build TorchTrainer as the Tune trainable ──────────────────────────────────
# HANDOFF: TorchTrainer receives train_ds and test_ds from Stage 2
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    scaling_config=ScalingConfig(
        num_workers=1,          # Colab-friendly; bump to 2-4 on a real cluster
        use_gpu=False,
    ),
    datasets={"train": train_ds},  # passed to all workers via train.get_dataset_shard()
)

tuner = tune.Tuner(
    trainer,
    param_space={"train_loop_config": search_space},
    tune_config=tune.TuneConfig(
        num_samples=5,       # 5 trials total (Colab budget)
        scheduler=scheduler,
        search_alg=searcher,
    ),
    run_config=train.RunConfig(
        name="breast-cancer-tune",
        stop={"val_accuracy": 0.98},  # stop early if we hit 98% accuracy
    ),
)

print("Starting Ray Tune search (5 trials × 8 epochs)...")
results = tuner.fit()
print("\nTune complete.")


### What just happened?
- **`tune.Tuner(trainer, ...)`** wraps `TorchTrainer` as a Tune trainable — Tune calls it once per trial, injecting `train_loop_config` as the `config` dict.
- **`ASHAScheduler`** implements the Asynchronous Successive Halving Algorithm: it terminates trials whose `val_accuracy` after the grace period is below the median, freeing resources for more promising configs.
- **`OptunaSearch`** uses Optuna's TPE (Tree-structured Parzen Estimator) to suggest `lr` values based on what worked in previous trials.
- **`num_samples=5`** with `grid_search([16, 32])` means the grid generates 2 base configs × sampled `lr` and `hidden_size` — Tune handles the combinatorics.


In [ ]:
import pandas as pd

# ── Retrieve best trial ───────────────────────────────────────────────────────
# HANDOFF: Tune results → best checkpoint for Stage 5 and Stage 6
best_result = results.get_best_result(metric="val_accuracy", mode="max")
best_config  = best_result.config["train_loop_config"]
best_metrics = best_result.metrics

print("Best hyperparameters:")
print(f"  lr          : {best_config['lr']:.6f}")
print(f"  batch_size  : {best_config['batch_size']}")
print(f"  hidden_size : {best_config['hidden_size']}")
print(f"\nBest metrics:")
print(f"  val_accuracy: {best_metrics['val_accuracy']:.4f}")
print(f"  val_loss    : {best_metrics['val_loss']:.4f}")

# Summary table of all trials
df = results.get_dataframe()
summary_cols = ["val_accuracy", "val_loss",
                "config/train_loop_config/lr",
                "config/train_loop_config/batch_size",
                "config/train_loop_config/hidden_size"]
available_cols = [c for c in summary_cols if c in df.columns]
print("\nAll trials summary:")
print(df[available_cols].sort_values("val_accuracy", ascending=False).to_string(index=False))


## Stage 5 · Reload Best Checkpoint — Inference on Test Set

The best checkpoint is stored in Ray's checkpoint storage. We reload it into a fresh `MLP` and run inference on the held-out test set.


In [ ]:
# ── Reload best checkpoint ────────────────────────────────────────────────────
best_checkpoint = best_result.checkpoint
checkpoint_data = best_checkpoint.to_dict()

hidden_size = checkpoint_data["hidden_size"]
input_dim   = checkpoint_data["input_dim"]

# Reconstruct the model with the best hyperparameters
best_model = MLP(input_dim=input_dim, hidden_size=hidden_size)
best_model.load_state_dict(checkpoint_data["model_state"])
best_model.eval()
print(f"Loaded best model: MLP(input_dim={input_dim}, hidden_size={hidden_size})")

# ── Run inference on test set ─────────────────────────────────────────────────
test_rows = list(test_ds.iter_rows())
X_test = torch.tensor([r["features"] for r in test_rows], dtype=torch.float32)
y_test = torch.tensor([r["label"]    for r in test_rows], dtype=torch.long)

with torch.no_grad():
    logits     = best_model(X_test)
    preds      = logits.argmax(dim=1)
    test_acc   = (preds == y_test).float().mean().item()
    proba      = torch.softmax(logits, dim=1)[:, 1]  # probability of malignant (class 1)

print(f"Test accuracy: {test_acc:.4f} on {len(y_test)} samples")
print(f"Predictions (first 10): {preds[:10].tolist()}")
print(f"True labels (first 10): {y_test[:10].tolist()}")

# Serialize model for Serve
import io
model_buffer = io.BytesIO()
torch.save({"model_state": best_model.state_dict(),
            "hidden_size": hidden_size,
            "input_dim"  : input_dim}, model_buffer)
model_bytes_serve = model_buffer.getvalue()
print(f"\nModel serialized: {len(model_bytes_serve):,} bytes")


### What just happened?
- **`best_result.checkpoint.to_dict()`** deserialises the checkpoint from Ray's object store — this is the Tune → Serve handoff artifact.
- We recreated the `MLP` with the exact `hidden_size` from the best trial — hyperparameter metadata stored in the checkpoint makes this reconstruction deterministic.
- **`model.eval()`** disables Dropout and BatchNorm training mode — always call this before inference.
- We serialize the model to bytes (`model_bytes_serve`) for passing into the Serve deployment via `bind()`.


## Stage 6 · Ray Serve — Deploy the Best Model as a REST API

**Handoff: best checkpoint → Ray Serve deployment**
We pass `model_bytes_serve` into the deployment via `.bind()`. The deployment loads the model once in `__init__` and serves predictions via `POST /cancer/predict`.


In [ ]:
from ray import serve
import requests as http_requests

# Start Serve (will reuse the existing Ray cluster)
serve.start(detached=False)
time.sleep(1)

@serve.deployment(
    num_replicas=1,
    ray_actor_options={"num_cpus": 0.5},
    max_concurrent_queries=10,
)
class CancerClassifier:
    """Serves the best Ray Tune checkpoint via HTTP POST /predict."""

    def __init__(self, model_bytes: bytes):
        # Load model once per replica at startup
        import io, torch
        buffer = io.BytesIO(model_bytes)
        data   = torch.load(buffer, weights_only=False)

        self.model = MLP(
            input_dim   = data["input_dim"],
            hidden_size = data["hidden_size"],
        )
        self.model.load_state_dict(data["model_state"])
        self.model.eval()
        self.class_names = ["benign", "malignant"]
        print("[CancerClassifier] Model loaded and ready.")

    async def __call__(self, request):
        """POST /predict — body: {\"features\": [[f1, f2, ..., f30]]}"""
        import torch
        body  = await request.json()
        feats = torch.tensor(body["features"], dtype=torch.float32)
        if feats.ndim == 1:
            feats = feats.unsqueeze(0)

        with torch.no_grad():
            logits = self.model(feats)
            preds  = logits.argmax(dim=1).tolist()
            proba  = torch.softmax(logits, dim=1).tolist()

        return {
            "predictions" : [self.class_names[p] for p in preds],
            "confidence"  : [round(max(p), 4) for p in proba],
            "class_indices": preds,
        }

# Deploy the application
serve_handle = serve.run(
    CancerClassifier.bind(model_bytes_serve),
    name="cancer-app",
    route_prefix="/cancer",
)
time.sleep(2)  # wait for replica to initialise
print("CancerClassifier deployed at http://localhost:8000/cancer")


### What just happened?
- **`model_bytes_serve`** is passed into `CancerClassifier.bind(...)` — Serve serialises this as an actor constructor argument; the model is deserialised in `__init__` on the worker node.
- **`max_concurrent_queries=10`** means each replica can handle 10 in-flight requests simultaneously, queuing the 11th until a slot frees.
- **`torch.no_grad()`** disables gradient tracking during inference — saves ~30% memory and speeds up forward passes by skipping autograd bookkeeping.
- The deployment is now reachable at `http://localhost:8000/cancer` — the next stage sends real HTTP requests to it.


## Stage 7 · Integration Test — Validate the Deployed Endpoint


In [ ]:
# ── Integration test: send 10 samples from the test set, check predictions ────

test_samples  = [r["features"] for r in test_rows[:10]]
true_labels   = [r["label"]    for r in test_rows[:10]]
class_names   = ["benign", "malignant"]

# Send all 10 samples in a single batch request
resp = http_requests.post(
    "http://localhost:8000/cancer",
    json={"features": test_samples},
    timeout=10,
)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

payload      = resp.json()
predictions  = payload["predictions"]
confidences  = payload["confidence"]
pred_indices = payload["class_indices"]

# Assert format correctness
assert len(predictions)  == 10, f"Expected 10 predictions, got {len(predictions)}"
assert len(confidences)  == 10, f"Expected 10 confidence scores"
assert all(p in class_names for p in predictions), "Unexpected class name"
assert all(0.0 <= c <= 1.0 for c in confidences), "Confidence out of [0,1]"

# Compare against true labels
matches = sum(pred_indices[i] == true_labels[i] for i in range(10))
print(f"Integration test PASSED ({matches}/10 correct predictions)")
print("\nSample predictions:")
for i in range(5):
    true_name = class_names[true_labels[i]]
    match_sym = "✓" if pred_indices[i] == true_labels[i] else "✗"
    print(f"  [{match_sym}] predicted={predictions[i]:<12} true={true_name:<12} confidence={confidences[i]:.3f}")


### What just happened?
- **`requests.post()`** sent a real HTTP request to the Serve endpoint — this is the same path a production client (mobile app, microservice) would use.
- The **assertions** are the integration test: format validation (length, class name membership, confidence range) plus accuracy against ground truth.
- A production integration test would also check latency (`resp.elapsed.total_seconds() < 0.5`), error handling (malformed input → 422), and load (concurrent requests via `concurrent.futures`).


## Stage 8 · Profile with `ray.timeline()` and Document Bottleneck


In [ ]:
# ── Capture the full pipeline timeline ────────────────────────────────────────
timeline_path = "/tmp/capstone_timeline.json"
ray.timeline(filename=timeline_path)

with open(timeline_path, "r") as f:
    trace = json.load(f)

events = trace if isinstance(trace, list) else trace.get("traceEvents", [])

# Find the top 5 longest duration events
duration_events = [
    ev for ev in events
    if ev.get("ph") in ("X", "B") and "dur" in ev
]
duration_events.sort(key=lambda e: e.get("dur", 0), reverse=True)

print(f"Total trace events: {len(events)}")
print(f"Duration events   : {len(duration_events)}")
print("\nTop 5 longest events:")
for ev in duration_events[:5]:
    dur_ms = ev.get("dur", 0) / 1000  # convert μs → ms
    print(f"  {ev.get('name','?'):<45} {dur_ms:8.1f} ms")

print(f"\nTimeline saved to: {timeline_path}")
print("Open at: https://ui.perfetto.dev")


## Bottleneck Analysis

Based on the timeline captured above, the typical bottleneck distribution in this capstone pipeline is:

| Stage | Expected duration | Typical bottleneck |
|---|---|---|
| `ray.init()` | 2–5 s | Cluster startup (one-time cost) |
| Ray Data preprocessing | 0.5–2 s | `map_batches` I/O from object store |
| **Ray Tune (5 trials × 8 epochs)** | **60–180 s** | **Dominant cost — training iterations** |
| Checkpoint reload | < 0.5 s | Deserialization from object store |
| Ray Serve replica startup | 1–3 s | Actor init + model load |
| HTTP inference (10 samples) | < 0.1 s | Negligible |

**Top bottleneck: Ray Tune training iterations**

The timeline will show multiple wide `train_loop_per_worker` bands — each representing one trial's full training run. The inter-trial gaps (thin vertical lines) are Tune's overhead for scheduling the next trial and loading its checkpoint.

**Optimization strategies for production:**
1. **Scale `num_workers`**: `ScalingConfig(num_workers=4)` distributes each epoch across 4 GPUs
2. **Increase `max_concurrent_trials`**: run 2–4 trials in parallel instead of sequentially
3. **Use `PBT` scheduler**: Population-Based Training mutates hyperparams mid-trial, avoiding wasted early epochs
4. **Pre-compile dataset**: use `ds.materialize()` to cache the preprocessed dataset in the object store before Tune starts


In [ ]:
# Challenge: Extend the pipeline with confidence thresholding
#
# The current deployment returns predictions for all inputs.
# Your task: modify the CancerClassifier (or create a new deployment)
# to implement confidence thresholding:
#
# 1. Add a CONFIDENCE_THRESHOLD = 0.85 class variable
# 2. In __call__, for any sample where max(softmax) < CONFIDENCE_THRESHOLD:
#    - Set the prediction to "uncertain" instead of a class name
#    - Add an "uncertain_indices" list to the response
# 3. Re-deploy and run the integration test again
# 4. Print how many of the 10 test samples fall below the threshold
#
# Scaffold:
# @serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.5})
# class ThresholdedClassifier:
#     CONFIDENCE_THRESHOLD = 0.85
#
#     def __init__(self, model_bytes: bytes):
#         ...  # same as CancerClassifier
#
#     async def __call__(self, request):
#         ...  # add thresholding logic here


In [ ]:
# Shutdown all Ray services cleanly
serve.shutdown()
ray.shutdown()
print("All Ray services shut down. Capstone complete!")


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `ray.cluster_resources()` | Total resources; compare with `available_resources()` to detect contention |
| Ray Data `from_items` + `map_batches` | Distributed dataset creation and vectorised preprocessing |
| **Handoff 1: Data → Train** | `datasets={"train": train_ds}` in `TorchTrainer`; workers call `get_dataset_shard` |
| `train.report({}, checkpoint=...)` | Sends metrics + checkpoint to Tune after each epoch |
| `ScalingConfig(num_workers=1)` | Single-worker training; bump `num_workers` for true distributed training |
| **Handoff 2: Train → Tune** | `TorchTrainer` passed as trainable to `tune.Tuner`; `param_space["train_loop_config"]` |
| `ASHAScheduler` | Prune bad trials early; `grace_period` controls minimum epochs before pruning |
| `OptunaSearch` | TPE sampler for intelligent search over continuous hyperparameter dims |
| **Handoff 3: Tune → Serve** | `best_result.checkpoint.to_dict()` → serialize → `Deployment.bind(model_bytes)` |
| Integration test | `requests.post()` + assertions on format, range, and accuracy |
| `ray.timeline()` | Profile; top bottleneck in this pipeline = Tune training iterations |

> **Tip:** Key handoff points: Ray Data → Ray Train (pass Dataset to TorchTrainer via `datasets={'train': ds}`), Ray Train → Ray Tune (pass TorchTrainer as trainable to Tuner), Ray Tune → Ray Serve (load best checkpoint from `tuner.get_results().get_best_result().checkpoint`).

---
## Congratulations!

You have completed the **Ray for Distributed Python** 10-day journey. You can now:
- Build distributed ML pipelines spanning data ingestion, training, hyperparameter search, and serving
- Scale each component independently using Ray's resource model
- Debug and profile distributed workloads with the Dashboard, timeline, and state APIs
- Deploy trained models as production REST endpoints with Ray Serve

Mark Day 10 complete in your [tracker](../index.html).
